In [ ]:
import pandas as pd
import fitz
import cv2
from io import BytesIO
from PIL import Image
import base64
from openai import OpenAI
from pydantic import BaseModel
import json
import numpy as np
import traceback
from typing import Optional
import io

import pandas as pd
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.styles import Alignment, Border, Side, Font

from blank_functions.forms.form_recognition import FormRecognition
from blank_functions.ui.ui_functions import get_pic_from_pdf, save_to_excel_local, get_correct_answers, postprocess_raw_output, check_answers, final_styling, extract_text_from_image, transform_json_to_dataframe
from blank_functions.ui.ui_functions import promt, prepare_cur_dict, reorder_cols

import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# no limit columns for pandas
pd.set_option('display.max_columns', None)

In [ ]:
import time

import os
# /Users/vladislav/Documents/mom_project_repo/streamlit-exam-form-app/notebooks/docling_test.ipynb
# os.chdir("C:\\Users\\zamko\\Documents\\mom_project\\repo\\notebooks")
# pdf_path = "./data/valid_format/valid_questions.pdf"
# answers_path = "./data/answers.xlsx"
# template_path = "./data/template_2.jpg"
# json_path = "./data/rows_data.json"

pdf_path = "C:/Users/zamko/Documents/mom_project/repo/data/valid_format/users_answears.pdf"
answers_path = "C:/Users/zamko/Documents/mom_project/repo/data/answers.xlsx"
template_path = "C:/Users/zamko/Documents/mom_project/repo/data/template_raw.jpg"
json_path = "C:/Users/zamko/Documents/mom_project/repo/data/rows_data_new_format.json"



pdf_bytes = open(pdf_path, 'rb').read()
pdf_document = fitz.open(stream=pdf_bytes, filetype="pdf")
num_pages = pdf_document.page_count

answers_bytes = open(answers_path, 'rb').read()
answers = pd.read_excel(BytesIO(answers_bytes))

cur_version = 2

df_global = pd.DataFrame()
form_dict = {}
for i in range(0, num_pages):
    cur_pic = get_pic_from_pdf(pdf_bytes, i, zoom=6.0)
    form = FormRecognition(
        image = cur_pic,
        template_path = template_path,
        json_path = json_path,
        answers = answers,
        version = cur_version)
    form = form.run_pipeline()
    cur_dict = prepare_cur_dict(form)
    df_current = transform_json_to_dataframe(cur_dict)
    df_global = pd.concat([df_global, df_current]).reset_index(drop=True)
    form_dict[i] = form

correct_answers = get_correct_answers(answers_bytes)
df_global_processed = postprocess_raw_output(df_global, correct_answers, cur_version)
df_global_answers = check_answers(df_global_processed)
df_global_styled = final_styling(df_global_answers)
df_global_styled = reorder_cols(df_global_styled)
save_to_excel_local(df_global_styled, form_dict)


In [ ]:
print(form_dict[0].answer7.cells[0].user_value)
print(form_dict[0].answer7.cells[1].user_value)
print(form_dict[0].answer7.cells[2].user_value)
print(form_dict[0].answer7.cells[3].user_value)


In [ ]:
print(form_dict[0].answer7.cells[0].correct_value)
print(form_dict[0].answer7.cells[1].correct_value)
print(form_dict[0].answer7.cells[2].correct_value)
print(form_dict[0].answer7.cells[3].correct_value)


In [5]:
form_dict[0].answer7.cells[2].user_value

In [ ]:
form_dict[0].answer7.cells[3].user_value

In [8]:
def prepare_cur_dict(form_dict):
    cur_dict = {}
    for row_name in ["date", "user_id", "version"]:
        row = getattr(form_dict, row_name)
        user_answers = row.user_answers
        cur_dict[row_name] = user_answers


    for row_name in [f"answer{i}" for i in range(1, 11)]:
        row = getattr(form_dict, row_name)
        user_answers = row.user_answers
        cur_dict[row_name] = user_answers


    for row_name in [f"correction{i}" for i in range(1, 11)]:
        row = getattr(form_dict, row_name)
        user_answers = row.user_answers
        cur_dict[row_name] = user_answers


    # for key in ["date", "user_id", "version"] + [f"answer{i}" for i in range(1, 11)] + [f"correction{i}" for i in range(1, 11)]:
    #     cur_dict[key] = "".join([x for x in cur_dict[key] if x is not None])


    # for key in [f"answer{i}" for i in range(1, 11)] + [f"correction{i}" for i in range(1, 11)]:
    #     if cur_dict[key] != '':
    #         cur_dict[key] = str(float(cur_dict[key].replace(',', '.')))

    # cur_dict['date'] = 'МАТЕМАТИКА'

    return cur_dict


# form = form.run_pipeline()
# cur_dict = prepare_cur_dict(form)
# df_current = transform_json_to_dataframe(cur_dict)
# df_global = pd.concat([df_global, df_current]).reset_index(drop=True)
# form_dict[i] = form

# correct_answers = get_correct_answers(answers_bytes)
# df_global_processed = postprocess_raw_output(df_global, correct_answers, cur_version)
# df_global_answers = check_answers(df_global_processed)
# df_global_styled = final_styling(df_global_answers)
# df_global_styled = reorder_cols(df_global_styled)
# save_to_excel_local(df_global_styled, form_dict)

In [ ]:
prepare_cur_dict(form)